In [3]:
import os
os.environ["JAVA_HOME"] = r"C:\Users\Ankkun\miniconda3\envs\aiEnv\Library"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
from pyspark.sql import SparkSession

In [4]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, datediff,
    max as spark_max, min as spark_min, lit, when, to_date
)
from pyspark.sql.utils import AnalysisException

In [5]:
jar1 = "file:///C:/Users/Ankkun/Documents/lap_trinh/my_project/streamlify/jars/hadoop-aws-3.3.4.jar"
jar2 = "file:///C:/Users/Ankkun/Documents/lap_trinh/my_project/streamlify/jars/aws-java-sdk-bundle-1.12.262.jar"


In [6]:

spark = SparkSession.builder.appName("Local_Notebook") \
    .config("spark.jars", f"{jar1},{jar2}") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9090") \
    .config("spark.hadoop.fs.s3a.access.key", "homura_madoka") \
    .config("spark.hadoop.fs.s3a.secret.key", "homura123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.socket.timeout", "60000") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")


In [7]:
BRONZE_PATH = "s3a://bronze-zone/datalake/raw/eventsim"
SILVER_PATH = "s3a://silver-zone/datalake/silver/eventsim"

In [8]:
df = spark.read.parquet(BRONZE_PATH)
print("Đọc xong, chưa action gì")

Đọc xong, chưa action gì


In [9]:
# ============================================================
# TEST CHURN LABEL MỚI — chạy trực tiếp trong notebook
# Yêu cầu: đã có SparkSession tên `spark` từ cell trước (đã config S3A xong)
# ============================================================

from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, datediff,
    max as spark_max, min as spark_min, lit, when, to_date
)
from pyspark.sql.utils import AnalysisException

SILVER_PATH = "s3a://silver-zone/datalake/silver/eventsim"


def build_user_features_test(silver_df, spark):
    """Feature engineering — bản test, in luôn kết quả ra màn hình thay vì ghi MinIO."""
    silver_df.createOrReplaceTempView("silver_events")
    max_date_row = spark.sql("SELECT MAX(event_time) AS max_dt FROM silver_events").collect()
    max_date = max_date_row[0]["max_dt"] if max_date_row and max_date_row[0]["max_dt"] else None

    # ── QUAN TRỌNG: lấy "level" tại event_time MỚI NHẤT của mỗi user, không groupBy theo level ──
    # Vì eventsim sinh random, user có thể đổi level nhiều lần (free<->paid),
    # groupBy("userId","level",...) trực tiếp sẽ tách 1 user thành nhiều dòng — sai cấu trúc.
    from pyspark.sql import Window
    from pyspark.sql.functions import row_number, desc

    w = Window.partitionBy("userId").orderBy(desc("event_time"))
    latest_level_df = (
        silver_df
        .withColumn("rn", row_number().over(w))
        .filter(col("rn") == 1)
        .select("userId", col("level").alias("current_level"), "gender")
    )

    user_features = silver_df.groupBy("userId").agg(
        count(when(col("page") == "NextSong", True)).alias("total_songs"),
        count(when(col("page") == "Thumbs Down", True)).alias("thumbs_down"),
        count(when(col("page") == "Thumbs Up", True)).alias("thumbs_up"),
        count(when(col("page") == "Settings", True)).alias("settings_visits"),
        count(when(col("page") == "Help", True)).alias("help_visits"),
        # Feature mới: số lần gặp Error — tín hiệu frustration, dễ liên quan churn
        count(when(col("page") == "Error", True)).alias("error_visits"),
        # Feature mới: tổng lượt tương tác xã hội (Add Friend, Add to Playlist) — engagement càng cao càng ít churn
        count(when(col("page").isin("Add Friend", "Add to Playlist"), True)).alias("social_actions"),
        # Signal churn THẬT — Downgrade tồn tại trong data của bạn, Cancellation Confirmation thì không
        count(when(col("page") == "Downgrade", True)).alias("downgrade_count"),
        count(when(col("page") == "Cancellation Confirmation", True)).alias("cancel_count"),
        count(when(col("page") == "Upgrade", True)).alias("upgrade_count"),
        count("sessionId").alias("total_sessions"),
        (datediff(to_date(lit(max_date)), to_date(spark_min("event_time"))) + 1).alias("days_active"),
    )

    # Join lại level hiện tại (mới nhất) + gender — 1 dòng / user, không trùng lặp
    user_features = user_features.join(latest_level_df, on="userId", how="left")

    user_features = user_features.withColumn(
        "dislike_ratio",
        when(col("total_songs") > 0, col("thumbs_down") / col("total_songs")).otherwise(0.0)
    ).withColumn(
        "avg_sessions_per_day",
        when(col("days_active") > 0, col("total_sessions") / col("days_active")).otherwise(0.0)
    ).withColumn(
        # Feature mới: tỷ lệ thích/không thích — khác dislike_ratio, đây là tỷ lệ TƯƠNG ĐỐI giữa 2 loại phản hồi
        "like_dislike_ratio",
        when(col("thumbs_down") > 0, col("thumbs_up") / col("thumbs_down")).otherwise(col("thumbs_up") + 1.0)
    ).withColumn(
        # Feature mới: mật độ lỗi gặp phải trên mỗi session — chuẩn hóa theo hoạt động, tránh user active nhiều bị tính sai
        "error_rate",
        when(col("total_sessions") > 0, col("error_visits") / col("total_sessions")).otherwise(0.0)
    )

    return user_features


def create_churn_label_test(user_features):
    """
    Label MỚI (v3 — chặt hơn): churn = ĐANG ở free VÀ đã từng downgrade/cancel.
    Lý do: eventsim random khiến user downgrade rồi upgrade lại liên tục (dao động bình thường,
    không phải rời bỏ thật). Chỉ tính churn nếu trạng thái HIỆN TẠI là free SAU KHI đã có
    hành vi downgrade/cancel — phản ánh đúng "đã rời và chưa quay lại tới giờ".
    """
    df_new = user_features.withColumn(
        "churn_new",
        when(
            (col("current_level") == "free") &
            ((col("cancel_count") > 0) | (col("downgrade_count") > 0)),
            lit(1)
        ).otherwise(lit(0))
    )
    # Label cũ (buggy) để so sánh
    df_compare = df_new.withColumn(
        "churn_old_buggy",
        when((col("cancel_count") > 0) | (col("current_level") == "free"), lit(1)).otherwise(lit(0))
    )
    return df_compare


def run_test(spark):
    print("📦 Đọc Silver layer...")
    silver_df = spark.read.parquet(SILVER_PATH).cache()
    print(f"   Tổng rows: {silver_df.count():,}")

    print("\n⚙️  Tính feature theo user...")
    user_features = build_user_features_test(silver_df, spark)

    print("\n🏷️  Gán label (cả label mới và cũ để so sánh)...")
    labeled_df = create_churn_label_test(user_features).cache()

    total_users = labeled_df.count()
    churn_new = labeled_df.filter(col("churn_new") == 1).count()
    churn_old = labeled_df.filter(col("churn_old_buggy") == 1).count()

    print(f"\n{'='*55}")
    print(f"  SO SÁNH LABEL CŨ (buggy) vs LABEL MỚI (đúng)")
    print(f"{'='*55}")
    print(f"  Tổng số user           : {total_users:>8,}")
    print(f"  Churn (label CŨ, buggy): {churn_old:>8,}  ({100*churn_old/total_users:.1f}%)")
    print(f"  Churn (label MỚI, đúng): {churn_new:>8,}  ({100*churn_new/total_users:.1f}%)")
    print(f"  Chênh lệch             : {churn_old - churn_new:>8,} user bị gán SAI ở label cũ")

    print(f"\n── Mẫu user bị gán churn=1 SAI ở label cũ (free hiện tại nhưng chưa từng downgrade/cancel) ──")
    labeled_df.filter(
        (col("churn_old_buggy") == 1) & (col("churn_new") == 0)
    ).select("userId", "current_level", "downgrade_count", "cancel_count", "upgrade_count", "churn_old_buggy", "churn_new") \
     .show(10, truncate=False)

    print(f"\n── Mẫu user đã từng Downgrade nhưng hiện tại lại là 'paid' (downgrade rồi upgrade lại) ──")
    labeled_df.filter(
        (col("downgrade_count") > 0) & (col("current_level") == "paid")
    ).select("userId", "current_level", "downgrade_count", "upgrade_count", "churn_new") \
     .show(10, truncate=False)

    print(f"\n── Phân phối current_level cho từng nhóm churn_new ──")
    labeled_df.groupBy("churn_new", "current_level").count().orderBy("churn_new", "current_level").show()

    silver_df.unpersist()
    labeled_df.unpersist()
    return labeled_df


# Cách dùng trong notebook:
labeled_df = run_test(spark)

📦 Đọc Silver layer...
   Tổng rows: 115,836

⚙️  Tính feature theo user...

🏷️  Gán label (cả label mới và cũ để so sánh)...

  SO SÁNH LABEL CŨ (buggy) vs LABEL MỚI (đúng)
  Tổng số user           :      201
  Churn (label CŨ, buggy):       93  (46.3%)
  Churn (label MỚI, đúng):       93  (46.3%)
  Chênh lệch             :        0 user bị gán SAI ở label cũ

── Mẫu user bị gán churn=1 SAI ở label cũ (free hiện tại nhưng chưa từng downgrade/cancel) ──
+------+-------------+---------------+------------+-------------+---------------+---------+
|userId|current_level|downgrade_count|cancel_count|upgrade_count|churn_old_buggy|churn_new|
+------+-------------+---------------+------------+-------------+---------------+---------+
+------+-------------+---------------+------------+-------------+---------------+---------+


── Mẫu user đã từng Downgrade nhưng hiện tại lại là 'paid' (downgrade rồi upgrade lại) ──
+------+-------------+---------------+-------------+---------+
|userId|current_leve

In [10]:
# ============================================================
# TRAIN CHURN MODEL — chạy trong notebook, KHÔNG ghi vào MinIO/Airflow
# Yêu cầu: đã chạy churn_label_test_v2.py trước, có sẵn `labeled_df` trong notebook
# (Nếu chưa có, chạy: exec(open("churn_label_test_v2.py").read()); labeled_df = run_test(spark))
# ============================================================

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline


def train_churn_notebook(labeled_df):
    """
    Train thử GBTClassifier ngay trong notebook để đánh giá nhanh, KHÔNG lưu model,
    KHÔNG ghi MinIO. Chỉ in ra metric để bạn quyết định có đáng đẩy vào batch job không.
    """
    print("🚀 Bắt đầu train thử (chỉ trong notebook, không lưu gì cả)...")

    # Dùng lại đúng churn_new đã verify ở bước trước — đổi tên cho gọn
    df = labeled_df.withColumnRenamed("churn_new", "churn")

    feature_cols = [
        "total_songs", "thumbs_down", "thumbs_up", "settings_visits",
        "help_visits", "total_sessions", "days_active",
        "dislike_ratio", "avg_sessions_per_day",
        # ĐÃ THỬ thêm error_visits, error_rate, social_actions, like_dislike_ratio (xem case study) —
        # AUC giảm 0.6333 -> 0.5273 vì mẫu chỉ 170 dòng train, thêm chiều làm tăng nhiễu (curse of
        # dimensionality), không phải feature đó vô giá trị về bản chất — chỉ là chưa đủ data
        # để hỗ trợ thêm. Quay lại feature set gọn cho tới khi có nhiều user/event hơn.
        # KHÔNG đưa downgrade_count/cancel_count/upgrade_count vào feature —
        # đây chính là thứ đã dùng để TẠO label, đưa vào feature sẽ là leakage trực tiếp 100%.
    ]

    gender_indexer = StringIndexer(inputCol="gender", outputCol="gender_idx", handleInvalid="keep")
    assembler = VectorAssembler(inputCols=feature_cols + ["gender_idx"], outputCol="features", handleInvalid="keep")

    gbt = GBTClassifier(
        featuresCol="features",
        labelCol="churn",
        maxIter=20,
        maxDepth=4,
        stepSize=0.1,
    )

    pipeline = Pipeline(stages=[gender_indexer, assembler, gbt])

    train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
    print(f"   Train: {train_df.count()} users | Test: {test_df.count()} users")

    model = pipeline.fit(train_df)
    predictions = model.transform(test_df)

    auc_eval = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC")
    auc = auc_eval.evaluate(predictions)

    acc_eval = MulticlassClassificationEvaluator(labelCol="churn", predictionCol="prediction", metricName="accuracy")
    f1_eval = MulticlassClassificationEvaluator(labelCol="churn", predictionCol="prediction", metricName="f1")
    acc = acc_eval.evaluate(predictions)
    f1 = f1_eval.evaluate(predictions)

    print(f"\n{'='*50}")
    print(f"  KẾT QUẢ TRAIN (chỉ để đánh giá, chưa lưu gì)")
    print(f"{'='*50}")
    print(f"  AUC      : {auc:.4f}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1-score : {f1:.4f}")

    print(f"\n── Confusion matrix (label thật vs dự đoán) ──")
    predictions.groupBy("churn", "prediction").count().orderBy("churn", "prediction").show()

    print(f"\n── Feature importance (cái nào ảnh hưởng nhiều nhất tới dự đoán) ──")
    gbt_model = model.stages[-1]
    importances = list(zip(feature_cols + ["gender_idx"], gbt_model.featureImportances.toArray()))
    for name, imp in sorted(importances, key=lambda x: -x[1]):
        print(f"   {name:<22} {imp:.4f}")

    return model, predictions


# Cách dùng trong notebook:
model, predictions = train_churn_notebook(labeled_df)

🚀 Bắt đầu train thử (chỉ trong notebook, không lưu gì cả)...
   Train: 170 users | Test: 31 users

  KẾT QUẢ TRAIN (chỉ để đánh giá, chưa lưu gì)
  AUC      : 0.5125
  Accuracy : 0.4516
  F1-score : 0.4482

── Confusion matrix (label thật vs dự đoán) ──
+-----+----------+-----+
|churn|prediction|count|
+-----+----------+-----+
|    0|       0.0|    8|
|    0|       1.0|    7|
|    1|       0.0|   10|
|    1|       1.0|    6|
+-----+----------+-----+


── Feature importance (cái nào ảnh hưởng nhiều nhất tới dự đoán) ──
   total_songs            0.1995
   dislike_ratio          0.1661
   settings_visits        0.1504
   total_sessions         0.1190
   help_visits            0.1182
   thumbs_down            0.1046
   thumbs_up              0.1003
   gender_idx             0.0419
   days_active            0.0000
   avg_sessions_per_day   0.0000


In [11]:
# ============================================================
# TRAIN CHURN MODEL — chạy trong notebook, KHÔNG ghi vào MinIO/Airflow
# Yêu cầu: đã chạy churn_label_test_v2.py trước, có sẵn `labeled_df` trong notebook
# (Nếu chưa có, chạy: exec(open("churn_label_test_v2.py").read()); labeled_df = run_test(spark))
# ============================================================

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline


def train_churn_notebook(labeled_df):
    """
    Train thử GBTClassifier ngay trong notebook để đánh giá nhanh, KHÔNG lưu model,
    KHÔNG ghi MinIO. Chỉ in ra metric để bạn quyết định có đáng đẩy vào batch job không.
    """
    print("🚀 Bắt đầu train thử (chỉ trong notebook, không lưu gì cả)...")

    # Dùng lại đúng churn_new đã verify ở bước trước — đổi tên cho gọn
    df = labeled_df.withColumnRenamed("churn_new", "churn")

    feature_cols = [
        "total_songs", "thumbs_down", "thumbs_up", "settings_visits",
        "help_visits", "total_sessions", "days_active",
        "dislike_ratio", "avg_sessions_per_day",
        # ĐÃ THỬ thêm error_visits, error_rate, social_actions, like_dislike_ratio (xem case study) —
        # AUC giảm 0.6333 -> 0.5273 vì mẫu chỉ 170 dòng train, thêm chiều làm tăng nhiễu (curse of
        # dimensionality), không phải feature đó vô giá trị về bản chất — chỉ là chưa đủ data
        # để hỗ trợ thêm. Quay lại feature set gọn cho tới khi có nhiều user/event hơn.
        # KHÔNG đưa downgrade_count/cancel_count/upgrade_count vào feature —
        # đây chính là thứ đã dùng để TẠO label, đưa vào feature sẽ là leakage trực tiếp 100%.
    ]

    gender_indexer = StringIndexer(inputCol="gender", outputCol="gender_idx", handleInvalid="keep")
    assembler = VectorAssembler(inputCols=feature_cols + ["gender_idx"], outputCol="features", handleInvalid="keep")

    gbt = GBTClassifier(
        featuresCol="features",
        labelCol="churn",
        maxIter=20,
        maxDepth=4,
        stepSize=0.1,
    )

    pipeline = Pipeline(stages=[gender_indexer, assembler, gbt])

    train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
    print(f"   Train: {train_df.count()} users | Test: {test_df.count()} users")

    model = pipeline.fit(train_df)
    predictions = model.transform(test_df)

    auc_eval = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC")
    auc = auc_eval.evaluate(predictions)

    acc_eval = MulticlassClassificationEvaluator(labelCol="churn", predictionCol="prediction", metricName="accuracy")
    f1_eval = MulticlassClassificationEvaluator(labelCol="churn", predictionCol="prediction", metricName="f1")
    acc = acc_eval.evaluate(predictions)
    f1 = f1_eval.evaluate(predictions)

    print(f"\n{'='*50}")
    print(f"  KẾT QUẢ TRAIN — seed=42 (chỉ 1 lần chia, xem thêm multi-seed dưới)")
    print(f"{'='*50}")
    print(f"  AUC      : {auc:.4f}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1-score : {f1:.4f}")

    print(f"\n── Confusion matrix (label thật vs dự đoán) ──")
    predictions.groupBy("churn", "prediction").count().orderBy("churn", "prediction").show()

    print(f"\n── Feature importance (cái nào ảnh hưởng nhiều nhất tới dự đoán) ──")
    gbt_model = model.stages[-1]
    importances = list(zip(feature_cols + ["gender_idx"], gbt_model.featureImportances.toArray()))
    for name, imp in sorted(importances, key=lambda x: -x[1]):
        print(f"   {name:<22} {imp:.4f}")

    # ── QUAN TRỌNG: đo độ ỔN ĐỊNH thật bằng cách chia lại nhiều seed khác nhau ──
    # Nếu AUC dao động mạnh giữa các seed -> xác nhận mẫu quá nhỏ, không phải do feature/code sai.
    print(f"\n{'='*50}")
    print(f"  ĐỘ ỔN ĐỊNH AUC QUA NHIỀU SEED KHÁC NHAU (multi-seed check)")
    print(f"{'='*50}")
    auc_list = []
    for seed in [1, 7, 13, 42, 99, 123, 2024]:
        tr, te = df.randomSplit([0.8, 0.2], seed=seed)
        m = pipeline.fit(tr)
        pred = m.transform(te)
        a = auc_eval.evaluate(pred)
        auc_list.append(a)
        print(f"   seed={seed:<5} AUC={a:.4f}  (test size={te.count()})")

    import statistics
    print(f"\n   AUC trung bình : {statistics.mean(auc_list):.4f}")
    print(f"   AUC độ lệch chuẩn (std) : {statistics.stdev(auc_list):.4f}")
    print(f"   AUC min-max    : {min(auc_list):.4f} - {max(auc_list):.4f}")
    if statistics.stdev(auc_list) > 0.08:
        print(f"   ⚠️  Std cao -> AUC dao động mạnh theo cách chia data, KHÔNG đáng tin với 1 lần chạy đơn lẻ.")
        print(f"      Đây là dấu hiệu mẫu (201 user) quá nhỏ để đánh giá ổn định, không phải lỗi code/feature.")

    return model, predictions


# Cách dùng trong notebook:
model, predictions = train_churn_notebook(labeled_df)

🚀 Bắt đầu train thử (chỉ trong notebook, không lưu gì cả)...
   Train: 170 users | Test: 31 users

  KẾT QUẢ TRAIN — seed=42 (chỉ 1 lần chia, xem thêm multi-seed dưới)
  AUC      : 0.5125
  Accuracy : 0.4516
  F1-score : 0.4482

── Confusion matrix (label thật vs dự đoán) ──
+-----+----------+-----+
|churn|prediction|count|
+-----+----------+-----+
|    0|       0.0|    8|
|    0|       1.0|    7|
|    1|       0.0|   10|
|    1|       1.0|    6|
+-----+----------+-----+


── Feature importance (cái nào ảnh hưởng nhiều nhất tới dự đoán) ──
   total_songs            0.1995
   dislike_ratio          0.1661
   settings_visits        0.1504
   total_sessions         0.1190
   help_visits            0.1182
   thumbs_down            0.1046
   thumbs_up              0.1003
   gender_idx             0.0419
   days_active            0.0000
   avg_sessions_per_day   0.0000

  ĐỘ ỔN ĐỊNH AUC QUA NHIỀU SEED KHÁC NHAU (multi-seed check)
   seed=1     AUC=0.5167  (test size=38)
   seed=7     AUC=0.6

In [12]:
# ============================================================
# TEST ĐẦY ĐỦ: XGBoost (có fallback) + Multi-seed validation
# Dùng ĐÚNG logic của churn_prediction.py thật, chạy trong notebook trước khi đẩy batch.
# Yêu cầu: đã có `labeled_df` từ churn_label_test_v2.py (cột churn_new là label dùng)
# ============================================================

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import statistics

FEATURE_COLS = [
    "total_songs", "thumbs_down", "thumbs_up", "settings_visits",
    "help_visits", "total_sessions", "days_active",
    "dislike_ratio", "avg_sessions_per_day",
]


def build_pipeline_with_xgboost_check():
    """
    Giống đúng logic trong churn_prediction.py thật:
    thử import XGBoost trước, nếu không có thì fallback GBTClassifier.
    In rõ ra đang dùng cái nào — để bạn biết chắc, không đoán mò.
    """
    gender_indexer = StringIndexer(inputCol="gender", outputCol="gender_idx", handleInvalid="keep")
    assembler = VectorAssembler(inputCols=FEATURE_COLS + ["gender_idx"], outputCol="features", handleInvalid="keep")

    try:
        from xgboost.spark import SparkXGBClassifier
        clf = SparkXGBClassifier(
            features_col="features",
            label_col="churn",
            num_workers=1,       # notebook local — chỉ 1 worker, khác Airflow dùng cluster
            use_gpu=False,
            n_estimators=50,
            max_depth=4,
            learning_rate=0.1,
            eval_metric="auc",
        )
        model_name = "XGBoost (SparkXGBClassifier) — THẬT"
    except ImportError:
        from pyspark.ml.classification import GBTClassifier
        clf = GBTClassifier(
            featuresCol="features",
            labelCol="churn",
            maxIter=20,
            maxDepth=4,
            stepSize=0.1,
        )
        model_name = "GBTClassifier (fallback) — KHÔNG phải XGBoost"

    print(f"📌 Model đang dùng: {model_name}")
    return Pipeline(stages=[gender_indexer, assembler, clf]), model_name


def run_full_validation(labeled_df, seeds=[1, 7, 13, 42, 99, 123, 2024]):
    """
    Test đầy đủ: kiểm tra XGBoost/GBT đang dùng cái nào, rồi chạy multi-seed
    để đo độ ổn định AUC thật — đúng quy trình trước khi quyết định đẩy batch.
    """
    df = labeled_df.withColumnRenamed("churn_new", "churn")

    pipeline, model_name = build_pipeline_with_xgboost_check()
    auc_eval = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC")

    print(f"\n{'='*55}")
    print(f"  MULTI-SEED VALIDATION — {model_name}")
    print(f"{'='*55}")

    auc_list = []
    for seed in seeds:
        train_df, test_df = df.randomSplit([0.8, 0.2], seed=seed)
        model = pipeline.fit(train_df)
        pred = model.transform(test_df)
        auc = auc_eval.evaluate(pred)
        auc_list.append(auc)
        print(f"   seed={seed:<5} AUC={auc:.4f}  (train={train_df.count()}, test={test_df.count()})")

    mean_auc = statistics.mean(auc_list)
    std_auc = statistics.stdev(auc_list)
    min_auc, max_auc = min(auc_list), max(auc_list)

    print(f"\n   AUC trung bình : {mean_auc:.4f}")
    print(f"   AUC std        : {std_auc:.4f}")
    print(f"   AUC min-max    : {min_auc:.4f} - {max_auc:.4f}")

    print(f"\n{'='*55}")
    print(f"  QUYẾT ĐỊNH: có nên đẩy vào batch job không?")
    print(f"{'='*55}")
    if std_auc > 0.08:
        print(f"   ❌ CHƯA NÊN. Std={std_auc:.4f} > 0.08 — AUC dao động quá mạnh theo cách chia data.")
        print(f"      Nguyên nhân nhiều khả năng: số user còn quá ít để model ổn định.")
        print(f"      Khuyến nghị: tăng NUM_USERS / chờ thêm data, KHÔNG đẩy batch lúc này.")
    elif mean_auc < 0.55:
        print(f"   ⚠️  CHƯA NÊN. AUC trung bình {mean_auc:.4f} gần mức random (0.5).")
        print(f"      Model ổn định (std thấp) nhưng KHÔNG học được tín hiệu đáng kể.")
        print(f"      Khuyến nghị: cần thêm feature có ý nghĩa hơn hoặc thêm data trước khi đẩy batch.")
    else:
        print(f"   ✅ CÓ THỂ ĐẨY. AUC trung bình {mean_auc:.4f}, std {std_auc:.4f} — ổn định và có tín hiệu.")
        print(f"      Nhớ cập nhật churn_prediction.py thật với đúng feature_cols đã verify ở đây.")

    return auc_list, model_name


# Cách dùng trong notebook (SAU KHI đã chạy churn_label_test_v2.py để có labeled_df):
auc_list, model_name = run_full_validation(labeled_df)

📌 Model đang dùng: XGBoost (SparkXGBClassifier) — THẬT

  MULTI-SEED VALIDATION — XGBoost (SparkXGBClassifier) — THẬT


2026-06-27 22:12:18,940 INFO XGBoost-PySpark: _fit Running xgboost-3.3.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'eval_metric': 'auc', 'learning_rate': 0.1, 'max_depth': 4, 'use_gpu': False, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}


TypeError: PandasMapOpsMixin.mapInPandas() got an unexpected keyword argument 'profile'

In [16]:
# ============================================================
# TEST ĐẦY ĐỦ: XGBoost (có fallback) + Multi-seed validation
# Dùng ĐÚNG logic của churn_prediction.py thật, chạy trong notebook trước khi đẩy batch.
# Yêu cầu: đã có `labeled_df` từ churn_label_test_v2.py (cột churn_new là label dùng)
# ============================================================

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import statistics

FEATURE_COLS_BASE = [
    "total_songs", "thumbs_down", "thumbs_up", "settings_visits",
    "help_visits", "total_sessions", "days_active",
    "dislike_ratio", "avg_sessions_per_day",
]

FEATURE_COLS_EXTENDED = FEATURE_COLS_BASE + [
    "error_visits", "error_rate", "social_actions", "like_dislike_ratio",
]


def build_pipeline_with_xgboost_check(feature_cols, force_gbt=True):
    """
    Giống đúng logic trong churn_prediction.py thật:
    thử import XGBoost trước, nếu không có thì fallback GBTClassifier.

    force_gbt=True (mặc định): LUÔN dùng GBTClassifier khi chạy trong notebook trên Windows.
    Lý do: XGBoost4J-Spark KHÔNG hỗ trợ Windows (training phân tán không hoạt động trên
    platform này) — dù import được, lúc .fit() sẽ lỗi version mismatch (mapInPandas 'profile').
    Đây không phải lỗi cấu hình cần sửa, mà là giới hạn nền tảng. Trong container Docker
    Linux thật (Airflow/Spark), code churn_prediction.py vẫn tự thử XGBoost trước như cũ.
    """
    gender_indexer = StringIndexer(inputCol="gender", outputCol="gender_idx", handleInvalid="keep")
    assembler = VectorAssembler(inputCols=feature_cols + ["gender_idx"], outputCol="features", handleInvalid="keep")

    if force_gbt:
        from pyspark.ml.classification import GBTClassifier
        clf = GBTClassifier(
            featuresCol="features",
            labelCol="churn",
            maxIter=20,
            maxDepth=4,
            stepSize=0.1,
        )
        model_name = "GBTClassifier (ép dùng — notebook chạy trên Windows, XGBoost-Spark không support)"
        print(f"📌 Model đang dùng: {model_name}")
        return Pipeline(stages=[gender_indexer, assembler, clf]), model_name

    try:
        from xgboost.spark import SparkXGBClassifier
        clf = SparkXGBClassifier(
            features_col="features",
            label_col="churn",
            num_workers=1,
            use_gpu=False,
            n_estimators=50,
            max_depth=4,
            learning_rate=0.1,
            eval_metric="auc",
        )
        model_name = "XGBoost (SparkXGBClassifier) — THẬT"
    except ImportError:
        from pyspark.ml.classification import GBTClassifier
        clf = GBTClassifier(
            featuresCol="features",
            labelCol="churn",
            maxIter=20,
            maxDepth=4,
            stepSize=0.1,
        )
        model_name = "GBTClassifier (fallback) — KHÔNG phải XGBoost"

    print(f"📌 Model đang dùng: {model_name}")
    return Pipeline(stages=[gender_indexer, assembler, clf]), model_name


def run_full_validation(labeled_df, feature_cols=FEATURE_COLS_BASE, seeds=[1, 7, 13, 42, 99, 123, 2024], label="BASE"):
    """
    Test đầy đủ: kiểm tra XGBoost/GBT đang dùng cái nào, rồi chạy multi-seed
    để đo độ ổn định AUC thật — đúng quy trình trước khi quyết định đẩy batch.
    """
    df = labeled_df.withColumnRenamed("churn_new", "churn") if "churn_new" in labeled_df.columns else labeled_df

    pipeline, model_name = build_pipeline_with_xgboost_check(feature_cols)
    auc_eval = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC")

    print(f"\n{'='*55}")
    print(f"  MULTI-SEED VALIDATION — [{label}] {len(feature_cols)} features — {model_name}")
    print(f"{'='*55}")

    auc_list = []
    for seed in seeds:
        train_df, test_df = df.randomSplit([0.8, 0.2], seed=seed)
        model = pipeline.fit(train_df)
        pred = model.transform(test_df)
        auc = auc_eval.evaluate(pred)
        auc_list.append(auc)
        print(f"   seed={seed:<5} AUC={auc:.4f}  (train={train_df.count()}, test={test_df.count()})")

    mean_auc = statistics.mean(auc_list)
    std_auc = statistics.stdev(auc_list)
    min_auc, max_auc = min(auc_list), max(auc_list)

    print(f"\n   AUC trung bình : {mean_auc:.4f}")
    print(f"   AUC std        : {std_auc:.4f}")
    print(f"   AUC min-max    : {min_auc:.4f} - {max_auc:.4f}")

    return auc_list, model_name, mean_auc, std_auc


def compare_base_vs_extended(labeled_df, seeds=[1, 7, 13, 42, 99, 123, 2024]):
    """
    Chạy CẢ HAI cấu hình feature (gốc 9 cột vs mở rộng 13 cột) qua ĐÚNG cùng 7 seed,
    để so sánh công bằng — trả lời thẳng câu hỏi: thêm feature có giúp ổn định hơn không,
    hay chỉ là 1 seed cụ thể làm nó trông tệ/tốt oan.
    """
    print("🔬 SO SÁNH: feature gốc (9 cột) vs feature mở rộng (13 cột) — cùng 7 seed\n")

    auc_base, name_base, mean_base, std_base = run_full_validation(
        labeled_df, feature_cols=FEATURE_COLS_BASE, seeds=seeds, label="BASE (9 cols)"
    )
    auc_ext, name_ext, mean_ext, std_ext = run_full_validation(
        labeled_df, feature_cols=FEATURE_COLS_EXTENDED, seeds=seeds, label="EXTENDED (13 cols)"
    )

    print(f"\n{'='*55}")
    print(f"  BẢNG SO SÁNH TỔNG KẾT")
    print(f"{'='*55}")
    print(f"  {'':20} {'Mean AUC':>10} {'Std':>8} {'Min':>8} {'Max':>8}")
    print(f"  {'BASE (9 cols)':20} {mean_base:>10.4f} {std_base:>8.4f} {min(auc_base):>8.4f} {max(auc_base):>8.4f}")
    print(f"  {'EXTENDED (13 cols)':20} {mean_ext:>10.4f} {std_ext:>8.4f} {min(auc_ext):>8.4f} {max(auc_ext):>8.4f}")

    print(f"\n{'='*55}")
    print(f"  QUYẾT ĐỊNH")
    print(f"{'='*55}")
    better = "EXTENDED" if mean_ext > mean_base else "BASE"
    more_stable = "EXTENDED" if std_ext < std_base else "BASE"
    print(f"  AUC trung bình cao hơn  : {better}")
    print(f"  Ổn định hơn (std thấp)  : {more_stable}")

    if std_base > 0.08 or std_ext > 0.08:
        print(f"\n  ⚠️  Cả 2 cấu hình đều có std cao (>0.08) — vấn đề KHÔNG nằm ở số lượng feature,")
        print(f"      mà ở kích thước mẫu (201 user). Thêm/bớt feature không giải quyết gốc rễ.")
        print(f"      Khuyến nghị: CHƯA đẩy batch job, chờ thêm user/data trước.")
    else:
        print(f"\n  ✅ Cả 2 cấu hình ổn định — có thể chọn cấu hình {better} để đẩy batch.")

    return {
        "base": {"auc_list": auc_base, "mean": mean_base, "std": std_base},
        "extended": {"auc_list": auc_ext, "mean": mean_ext, "std": std_ext},
    }


# Cách dùng trong notebook (SAU KHI đã chạy churn_label_test_v2.py để có labeled_df):
results = compare_base_vs_extended(labeled_df)

🔬 SO SÁNH: feature gốc (9 cột) vs feature mở rộng (13 cột) — cùng 7 seed

📌 Model đang dùng: GBTClassifier (ép dùng — notebook chạy trên Windows, XGBoost-Spark không support)

  MULTI-SEED VALIDATION — [BASE (9 cols)] 9 features — GBTClassifier (ép dùng — notebook chạy trên Windows, XGBoost-Spark không support)
   seed=1     AUC=0.5167  (train=163, test=38)
   seed=7     AUC=0.6125  (train=160, test=41)
   seed=13    AUC=0.5357  (train=167, test=34)
   seed=42    AUC=0.5125  (train=170, test=31)
   seed=99    AUC=0.4377  (train=159, test=42)
   seed=123   AUC=0.4928  (train=159, test=42)
   seed=2024  AUC=0.5826  (train=158, test=43)

   AUC trung bình : 0.5272
   AUC std        : 0.0577
   AUC min-max    : 0.4377 - 0.6125
📌 Model đang dùng: GBTClassifier (ép dùng — notebook chạy trên Windows, XGBoost-Spark không support)

  MULTI-SEED VALIDATION — [EXTENDED (13 cols)] 13 features — GBTClassifier (ép dùng — notebook chạy trên Windows, XGBoost-Spark không support)
   seed=1     AUC=0.44